# Monitorización de Compliance Regulatorio

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Analizar textos regulatorios** con `CORTEX.COMPLETE()`
2. **Clasificar obligaciones** por tipología y urgencia
3. **Crear un sistema de alertas** de incumplimiento
4. **Generar resúmenes ejecutivos** de normativas con IA
5. **Monitorizar gaps de cumplimiento** con dashboard interactivo

---

## Lo Que Construirás

Un **sistema de compliance regulatorio** automatizado:
- Análisis automático de textos normativos con LLM
- Clasificación de obligaciones por área (capital, liquidez, conducta, PBC)
- Detección de gaps entre obligaciones y controles implementados
- Alertas de vencimiento y priorización de acciones
- Dashboard de estado de cumplimiento

**Valor de Negocio**: Reducir riesgo de multas regulatorias y automatizar la monitorización continua.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex AI habilitado
- **Aproximadamente 15 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave

- `CORTEX.COMPLETE()` — Análisis y resumen de textos regulatorios
- `CORTEX.SENTIMENT()` — Detectar tono de urgencia en circulares
- `GENERATOR()` — Datos sintéticos de obligaciones
- CTEs y `CASE` — Lógica de clasificación y priorización

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Compliance Bancario

**Objetivo**: Automatizar la monitorización de cumplimiento regulatorio en banca.

### Reguladores y Normativas Clave

| Regulador | Normativa | Área |
|---|---|---|
| BCE/SSM | CRR/CRD | Capital y Liquidez |
| EBA | ITS/RTS | Reporting regulatorio |
| CNMV | MiFID II | Conducta e inversión |
| SEPBLAC | 6AMLD | Prevención blanqueo |
| BdE | Circular 4/2017 | Contabilidad |

In [ ]:
CREATE DATABASE IF NOT EXISTS COMPLIANCE_REG_DB;
CREATE SCHEMA IF NOT EXISTS COMPLIANCE_REG_DB.PUBLIC;
USE SCHEMA COMPLIANCE_REG_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

---

## Paso 2: Definir Estructura de Datos

### Tablas

1. **`OBLIGACIONES_REG`** — Obligaciones regulatorias con texto normativo
2. **`CONTROLES`** — Controles implementados para cada obligación
3. **`EVALUACIONES`** — Evaluaciones periódicas de cumplimiento

In [ ]:
CREATE OR REPLACE TABLE OBLIGACIONES_REG (
    obligacion_id VARCHAR(10) PRIMARY KEY,
    regulador VARCHAR(20),
    normativa VARCHAR(50),
    area VARCHAR(30),
    titulo VARCHAR(200),
    descripcion TEXT,
    fecha_entrada_vigor DATE,
    fecha_limite_cumplimiento DATE,
    severidad VARCHAR(10),
    estado VARCHAR(20)
);

CREATE OR REPLACE TABLE CONTROLES (
    control_id VARCHAR(10) PRIMARY KEY,
    obligacion_id VARCHAR(10),
    nombre_control VARCHAR(200),
    tipo_control VARCHAR(30),
    responsable VARCHAR(100),
    frecuencia VARCHAR(20),
    ultima_revision DATE,
    resultado_ultima VARCHAR(20),
    FOREIGN KEY (obligacion_id) REFERENCES OBLIGACIONES_REG(obligacion_id)
);

CREATE OR REPLACE TABLE EVALUACIONES (
    evaluacion_id VARCHAR(10) PRIMARY KEY,
    obligacion_id VARCHAR(10),
    fecha_evaluacion DATE,
    evaluador VARCHAR(100),
    resultado VARCHAR(20),
    observaciones TEXT,
    acciones_correctivas TEXT,
    FOREIGN KEY (obligacion_id) REFERENCES OBLIGACIONES_REG(obligacion_id)
);

SELECT 'Tablas creadas' AS status;

---

## Paso 3: Generar Datos de Compliance

### Qué Vamos a Crear

- **50 obligaciones regulatorias** de 5 reguladores
- **150 controles** asociados (3 por obligación promedio)
- **200 evaluaciones** históricas con resultados

### Estados de Cumplimiento

- **Cumple**: Todos los controles operativos y efectivos
- **Parcial**: Algunos controles pendientes de implementar
- **No Cumple**: Gaps significativos identificados
- **En Progreso**: Implementación en curso

In [ ]:
-- Generar obligaciones regulatorias
INSERT INTO OBLIGACIONES_REG
WITH regs AS (
    SELECT column1 AS regulador, column2 AS normativa, column3 AS area FROM VALUES
    ('BCE','CRR III','Capital'),('BCE','LCR/NSFR','Liquidez'),('EBA','ITS Reporting','Reporting'),
    ('CNMV','MiFID II','Conducta'),('SEPBLAC','6AMLD','PBC/FT'),('BdE','Circular 4/2017','Contabilidad'),
    ('BCE','IRRBB','Riesgo Tipo Interés'),('EBA','DORA','Resiliencia Digital'),
    ('CNMV','SFDR','Sostenibilidad'),('SEPBLAC','Ley 10/2010','PBC/FT')
)
SELECT
    'OBL' || LPAD(SEQ4()::STRING, 4, '0'),
    r.regulador, r.normativa, r.area,
    'Obligación ' || r.normativa || ' #' || SEQ4(),
    'Requisito regulatorio según ' || r.normativa || ' del ' || r.regulador || '. Establece obligaciones de ' || r.area || ' para entidades de crédito supervisadas.',
    DATEADD('day', -UNIFORM(30, 730, RANDOM()), CURRENT_DATE()),
    DATEADD('day', UNIFORM(30, 365, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Alta','Media','Baja')[UNIFORM(0,2,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Cumple','Parcial','No Cumple','En Progreso')[UNIFORM(0,3,RANDOM())]::VARCHAR
FROM TABLE(GENERATOR(ROWCOUNT => 5)) CROSS JOIN regs r
WHERE SEQ4() <= 50;

-- Generar controles
INSERT INTO CONTROLES
SELECT
    'CTL' || LPAD(SEQ4()::STRING, 5, '0'),
    'OBL' || LPAD(UNIFORM(1, 50, RANDOM())::STRING, 4, '0'),
    'Control de ' || ARRAY_CONSTRUCT('Monitorización','Reporting','Validación','Auditoría','Testeo')[UNIFORM(0,4,RANDOM())]::VARCHAR || ' #' || SEQ4(),
    ARRAY_CONSTRUCT('Preventivo','Detectivo','Correctivo')[UNIFORM(0,2,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Dir. Riesgos','Dir. Compliance','Dir. Auditoría','Dir. Operaciones')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Diario','Semanal','Mensual','Trimestral')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    DATEADD('day', -UNIFORM(1, 90, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Efectivo','Parcial','Inefectivo')[UNIFORM(0,2,RANDOM())]::VARCHAR
FROM TABLE(GENERATOR(ROWCOUNT => 150));

-- Generar evaluaciones
INSERT INTO EVALUACIONES
SELECT
    'EVL' || LPAD(SEQ4()::STRING, 5, '0'),
    'OBL' || LPAD(UNIFORM(1, 50, RANDOM())::STRING, 4, '0'),
    DATEADD('day', -UNIFORM(1, 365, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Auditoría Interna','Compliance','Regulador','Externa')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Satisfactorio','Parcial','Insatisfactorio')[UNIFORM(0,2,RANDOM())]::VARCHAR,
    'Evaluación periódica de cumplimiento normativo.',
    CASE WHEN UNIFORM(0,2,RANDOM()) = 0 THEN 'Implementar controles adicionales' ELSE NULL END
FROM TABLE(GENERATOR(ROWCOUNT => 200));

SELECT 'Datos generados' AS status,
    (SELECT COUNT(*) FROM OBLIGACIONES_REG) AS obligaciones,
    (SELECT COUNT(*) FROM CONTROLES) AS controles,
    (SELECT COUNT(*) FROM EVALUACIONES) AS evaluaciones;

---

## Paso 4: Análisis de Textos Regulatorios con IA

### Qué Vamos a Hacer

Usar `CORTEX.COMPLETE()` para:
1. **Resumir** cada obligación regulatoria
2. **Clasificar** por nivel de impacto
3. **Identificar** acciones concretas requeridas

### ¿Por Qué LLM para Compliance?

- Las normativas bancarias son **extensas y complejas**
- Un LLM puede **extraer las acciones clave** rápidamente
- Permite **escalar** el análisis a cientos de normativas
- **Consistencia** en la interpretación vs análisis manual

In [ ]:
-- Análisis IA de obligaciones
CREATE OR REPLACE TABLE ANALISIS_OBLIGACIONES AS
SELECT
    obligacion_id,
    regulador,
    normativa,
    area,
    titulo,
    estado,
    severidad,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Analiza esta obligación regulatoria bancaria y genera un resumen de 3 líneas con las acciones concretas requeridas:\n' ||
        'Regulador: ' || regulador || '\n' ||
        'Normativa: ' || normativa || '\n' ||
        'Área: ' || area || '\n' ||
        'Descripción: ' || descripcion || '\n' ||
        'Responde solo en español con las acciones requeridas.'
    ) AS resumen_ia
FROM OBLIGACIONES_REG
SAMPLE (20 ROWS);

SELECT obligacion_id, regulador, normativa, area, resumen_ia
FROM ANALISIS_OBLIGACIONES LIMIT 5;

---

## Paso 5: Detectar Gaps de Cumplimiento

### Indicadores de Gap

- Obligaciones sin controles asociados
- Controles con resultado "Inefectivo"
- Evaluaciones insatisfactorias recientes
- Obligaciones con fecha límite próxima y estado "No Cumple"

In [ ]:
-- Identificar gaps de cumplimiento
CREATE OR REPLACE TABLE GAPS_COMPLIANCE AS
WITH control_stats AS (
    SELECT
        obligacion_id,
        COUNT(*) AS num_controles,
        SUM(CASE WHEN resultado_ultima = 'Efectivo' THEN 1 ELSE 0 END) AS controles_efectivos,
        SUM(CASE WHEN resultado_ultima = 'Inefectivo' THEN 1 ELSE 0 END) AS controles_inefectivos
    FROM CONTROLES GROUP BY 1
),
eval_stats AS (
    SELECT
        obligacion_id,
        COUNT(*) AS num_evaluaciones,
        SUM(CASE WHEN resultado = 'Insatisfactorio' THEN 1 ELSE 0 END) AS evals_insatisfactorias,
        MAX(fecha_evaluacion) AS ultima_evaluacion
    FROM EVALUACIONES GROUP BY 1
)
SELECT
    o.obligacion_id, o.regulador, o.normativa, o.area, o.severidad, o.estado,
    o.fecha_limite_cumplimiento,
    DATEDIFF('day', CURRENT_DATE(), o.fecha_limite_cumplimiento) AS dias_hasta_limite,
    COALESCE(cs.num_controles, 0) AS num_controles,
    COALESCE(cs.controles_efectivos, 0) AS controles_ok,
    COALESCE(cs.controles_inefectivos, 0) AS controles_ko,
    COALESCE(es.evals_insatisfactorias, 0) AS evals_ko,
    CASE
        WHEN o.estado = 'No Cumple' AND dias_hasta_limite < 30 THEN 'CRITICO'
        WHEN cs.controles_inefectivos > 0 OR es.evals_insatisfactorias > 0 THEN 'ALTO'
        WHEN o.estado = 'Parcial' THEN 'MEDIO'
        ELSE 'BAJO'
    END AS nivel_riesgo_gap
FROM OBLIGACIONES_REG o
LEFT JOIN control_stats cs ON o.obligacion_id = cs.obligacion_id
LEFT JOIN eval_stats es ON o.obligacion_id = es.obligacion_id;

SELECT nivel_riesgo_gap, COUNT(*) AS n, AVG(dias_hasta_limite) AS dias_promedio
FROM GAPS_COMPLIANCE GROUP BY 1 ORDER BY 1;

---

## Paso 6: Dashboard de Compliance

### Panel de Monitorización

Dashboard con KPIs de cumplimiento, mapa de calor por área regulatoria y tabla de gaps prioritarios.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Monitorización de Compliance Regulatorio')
st.markdown('### Estado de Cumplimiento Normativo')

df = session.sql('SELECT * FROM GAPS_COMPLIANCE').to_pandas()

total = len(df)
criticos = len(df[df['NIVEL_RIESGO_GAP'] == 'CRITICO'])
altos = len(df[df['NIVEL_RIESGO_GAP'] == 'ALTO'])
cumple = len(df[df['ESTADO'] == 'Cumple'])

c1, c2, c3, c4 = st.columns(4)
c1.metric('Total Obligaciones', total)
c2.metric('Gaps Críticos', criticos)
c3.metric('Gaps Altos', altos)
c4.metric('Tasa Cumplimiento', f'{cumple/total*100:.0f}%')

st.markdown('---')
st.subheader('Distribución por Nivel de Riesgo')
df_risk = df['NIVEL_RIESGO_GAP'].value_counts()
st.bar_chart(df_risk)

st.markdown('---')
st.subheader('Por Área Regulatoria')
df_area = df.groupby('AREA')['NIVEL_RIESGO_GAP'].value_counts().unstack(fill_value=0)
st.dataframe(df_area)

st.markdown('---')
st.subheader('Gaps Prioritarios')
df_gaps = session.sql("SELECT obligacion_id, regulador, normativa, area, severidad, estado, dias_hasta_limite, nivel_riesgo_gap FROM GAPS_COMPLIANCE WHERE nivel_riesgo_gap IN ('CRITICO','ALTO') ORDER BY dias_hasta_limite").to_pandas()
st.dataframe(df_gaps)

---

## Paso 7: Limpieza del Entorno (Opcional)

⚠️ **Advertencia**: Eliminará todos los datos.

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS COMPLIANCE_REG_DB;